# [2/4] NHÁNH RNN – GRU / LSTM / BiLSTM (+ BiGRU, stacked, optimizer)
### Đồ án Deep Learning · Chạy SAU file 01 (độc lập, cùng dataset & split seed 42)

Ảnh crop 128×128×3 → **chuỗi 128 bước × 384 chiều** (row-as-sequence, Session 05). Softmax 10 lớp, class weights.

HPO 3 vòng: (1) **cell × hidden** (GRU/LSTM/BiGRU/BiLSTM × 128/256 = 8 cấu hình); (2) **dropout & optimizer** trên cấu hình thắng (Adam vs RMSprop × dropout 0.2/0.4); (3) **stacked 2 tầng** của cấu hình thắng. → train cuối 15 epochs.

Kaggle Settings: **GPU T4 x2** + **+ Add Input dataset `ppe-unified-v1`** (Internet không bắt buộc — RNN tự xây, không dùng pretrained). Ước tính ~3h.

## Dataset: PPE_Unified_v1 (dùng chung cho cả 3 notebook — GIỮ NGUYÊN CELL NÀY Ở CẢ 3 FILE)
**Safety-PPE Unified v1** — dataset tự xây dựng bằng cách gộp 3 dataset public (CC BY 4.0): Hard Hat v2 (computer-vision-filqz) + PPE v1 (testcasque) + PPE v5 (mohamed-traore-2ekkp). **12 lớp = 6 cặp tuân thủ/vi phạm**, ~18k ảnh đã: khử trùng lặp theo ảnh gốc, khử trùng chéo MD5, **chuẩn hóa 640×640 letterbox**, chia sẵn **train 70% / val 20% / test 10% theo nhóm ảnh gốc (chống leakage)**. Chi tiết trong `DATASET_CARD.md` của dataset.

**Cách nạp:** dataset đã upload lên Kaggle (`ppe-unified-v1`); ở MỖI notebook bấm **+ Add Input → Your Datasets → ppe-unified-v1**. Không cần API key — dữ liệu nạp offline từ Input. (Lưu ý: Internet ON vẫn cần ở notebook 01/03 để cài thư viện + tải trọng số pretrained.)

Split đã cố định sẵn trong dataset ⇒ cả 3 notebook so sánh trên đúng cùng một phép chia. Xử lý mất cân bằng khi train: (1) repeat-factor oversampling lớp hiếm (train only); (2) undersampling crop ≤ MAX_CROPS_PER_CLASS; (3) class weights trong loss; (4) chọn model theo macro-F1 / mAP từng lớp (lớp có <10 mẫu test — vd no_boots — bị loại khỏi worst_class_f1, báo cáo riêng). Quy tắc: mọi hyperparameter chọn trên VAL; TEST chỉ dùng 1 lần cuối.

In [ ]:
# ====================== CONFIG (RNN) ======================
BRANCH     = "RNN"
QUICK_TEST = False    # True: chạy nhanh kiểm tra pipeline (~15-20 phút)
RUN_HPO    = True

SEED = 42
SPLIT = (0.7, 0.2, 0.1)
RFS_T, REPEAT_CAP = 0.15, 4                 # repeat-factor oversampling lớp hiếm (train only)
CROP_SIZE, CROP_SIZE_VIT = 128, 224
MAX_CROPS_PER_CLASS = 200 if QUICK_TEST else 2500

import os, random, time, shutil, json, numpy as np
random.seed(SEED); np.random.seed(SEED)
WORK = "/kaggle/working"
os.makedirs(f"{WORK}/results", exist_ok=True)
HPO_TABLES = {}

# ============ RESUME: đặt đường dẫn output của lần chạy trước của CHÍNH notebook này (nếu có) ============
RESUME_FROM = ""
if RESUME_FROM and os.path.exists(RESUME_FROM):
    for sub in ["state.json", "results", "runs"]:
        src = os.path.join(RESUME_FROM, sub)
        if os.path.isdir(src): shutil.copytree(src, os.path.join(WORK, sub), dirs_exist_ok=True)
        elif os.path.isfile(src): shutil.copy(src, os.path.join(WORK, sub))
    print("ĐÃ NẠP CHECKPOINT từ:", RESUME_FROM)

STATE_PATH = f"{WORK}/state.json"
STATE = json.load(open(STATE_PATH)) if os.path.exists(STATE_PATH) else {}
def save_state(): json.dump(STATE, open(STATE_PATH, "w"), indent=1, default=str)
def done(key):    return key in STATE
def mark(key, value=True): STATE[key] = value; save_state()
def free():
    import gc, torch; gc.collect(); torch.cuda.empty_cache()

print(f"[{BRANCH}] QUICK_TEST={QUICK_TEST} | RUN_HPO={RUN_HPO} | stage đã xong:", [k for k in STATE] or "(chưa có)")

In [ ]:
import torch
assert torch.cuda.is_available(), "Chưa bật GPU! Settings -> Accelerator -> GPU T4 x2"
cap = torch.cuda.get_device_capability(0); gpu_name = torch.cuda.get_device_name(0)
if cap[0] < 7:
    raise SystemError(f"GPU {gpu_name} (sm_{cap[0]}{cap[1]}) không được PyTorch hỗ trợ. Đổi sang GPU T4 x2!")
print("GPU:", gpu_name, "- OK")

In [ ]:
from pathlib import Path
from collections import Counter
import yaml, glob

# ==== Tìm PPE_Unified_v1 trong /kaggle/input (đã Add Input) ====
DS_ROOT = None
for c in glob.glob("/kaggle/input/**/data.yaml", recursive=True):
    root = os.path.dirname(c)
    if os.path.isdir(f"{root}/train/images") and os.path.isdir(f"{root}/val/images") and os.path.isdir(f"{root}/test/images"):
        DS_ROOT = root; break
assert DS_ROOT, ("Không tìm thấy PPE_Unified_v1! Upload thư mục PPE_Unified_v1 lên Kaggle "
                 "(Create -> New Dataset) rồi '+ Add Input' vào notebook này.")
print("Dataset input:", DS_ROOT)

raw_yaml = yaml.safe_load(open(f"{DS_ROOT}/data.yaml"))
raw_names = raw_yaml["names"]
if isinstance(raw_names, dict): raw_names = [raw_names[k] for k in sorted(raw_names)]
CLASSES = list(raw_names); NUM_CLS = len(CLASSES)
print(f"{NUM_CLS} lớp:", CLASSES)

# ==== Copy về /kaggle/working (input read-only; YOLO cần ghi labels.cache) ====
# Split ĐÃ cố định sẵn trong dataset (chia theo nhóm ảnh gốc, chống leakage) -> KHÔNG chia lại.
DATA = f"{WORK}/dataset"
CAPS = {"train": 300, "val": 80, "test": 40} if QUICK_TEST else {}
splits = {}
for sp in ["train", "val", "test"]:
    os.makedirs(f"{DATA}/{sp}/images", exist_ok=True); os.makedirs(f"{DATA}/{sp}/labels", exist_ok=True)
    files = sorted(os.listdir(f"{DS_ROOT}/{sp}/images"))
    random.Random(SEED).shuffle(files)
    if sp in CAPS: files = files[:CAPS[sp]]
    done_copy = len(os.listdir(f"{DATA}/{sp}/images")) >= len(files)   # resume: đã copy rồi thì bỏ qua
    for fn in files:
        if done_copy: break
        shutil.copy(f"{DS_ROOT}/{sp}/images/{fn}", f"{DATA}/{sp}/images/{fn}")
        lp = f"{DS_ROOT}/{sp}/labels/{Path(fn).stem}.txt"
        if os.path.exists(lp): shutil.copy(lp, f"{DATA}/{sp}/labels/{Path(fn).stem}.txt")
    splits[sp] = files
print("Số ảnh:", {sp: len(v) for sp, v in splits.items()})

# ==== Kiểm tra đồng nhất kích thước ảnh (dataset đã letterbox 640x640 khi build) ====
from PIL import Image
sample = [f"{DATA}/train/images/{f}" for f in splits["train"][:200]]
dims = Counter(Image.open(p).size for p in sample)
print("Kích thước ảnh (mẫu 200):", dict(dims.most_common(3)))
assert dims.most_common(1)[0][0] == (640, 640), "Ảnh chưa đồng nhất 640x640 - hãy build lại dataset với NORMALIZE_SIZE=640!"

# ==== Thống kê lớp + Repeat-factor oversampling (LVIS) - CHỈ tập train ====
cls_count, img_classes = Counter(), {}
for sp in ["train", "val", "test"]:
    for lf in os.listdir(f"{DATA}/{sp}/labels"):
        if "_rfs" in lf: continue
        present = Counter()
        for line in open(f"{DATA}/{sp}/labels/{lf}"):
            if not line.strip(): continue
            c = int(line.split()[0]); present[c] += 1; cls_count[(sp, CLASSES[c])] += 1
        if sp == "train": img_classes[lf] = present

from math import sqrt
n_train = len(splits["train"])
f_c = {c: sum(1 for s in img_classes.values() if c in s)/max(1, n_train) for c in range(NUM_CLS)}
r_c = {c: max(1.0, sqrt(RFS_T/max(f_c[c], 1e-6))) for c in range(NUM_CLS)}
print("Hệ số lặp lớp hiếm:", {CLASSES[c]: round(r_c[c],1) for c in range(NUM_CLS) if r_c[c] > 1.5})
for c in range(NUM_CLS):
    if cls_count[("train", CLASSES[c])]: cls_count[("train_after_rfs", CLASSES[c])] = cls_count[("train", CLASSES[c])]
added = 0
for lf, present in list(img_classes.items()):
    if not present: continue
    r = min(REPEAT_CAP, max(r_c[c] for c in present))
    if r < 1.5: continue
    stem = Path(lf).stem
    imgs = glob.glob(f"{DATA}/train/images/{stem}.*")
    if not imgs: continue
    ext = Path(imgs[0]).suffix
    for k in range(int(round(r)) - 1):
        dst = f"{DATA}/train/images/{stem}_rfs{k}{ext}"
        if not os.path.exists(dst):
            shutil.copy(imgs[0], dst)
            shutil.copy(f"{DATA}/train/labels/{lf}", f"{DATA}/train/labels/{stem}_rfs{k}.txt")
        for c, cnt in present.items(): cls_count[("train_after_rfs", CLASSES[c])] += cnt
        added += 1
print(f"Oversampling: +{added} bản sao ảnh chứa lớp hiếm (train {n_train} -> {n_train+added})")

names_yaml = "\n".join(f"  {i}: {nm}" for i, nm in enumerate(CLASSES))
with open(f"{DATA}/data.yaml", "w") as f:
    f.write(f"path: {DATA}\ntrain: train/images\nval: val/images\ntest: test/images\nnames:\n{names_yaml}\n")

import pandas as pd
df_dist = pd.DataFrame([{"split":k[0],"class":k[1],"boxes":v} for k,v in cls_count.items()]).pivot_table(index="class",columns="split",values="boxes",fill_value=0)
df_dist.to_csv(f"{WORK}/results/dataset_distribution.csv"); df_dist

In [ ]:
import cv2
from PIL import Image

def build_crops(split, size):
    X, y = [], []; per_class = Counter()
    files = sorted(os.listdir(f"{DATA}/{split}/labels"))
    random.Random(SEED).shuffle(files)
    for lf in files:
        stem = Path(lf).stem
        if "_rfs" in stem: continue
        for ext in [".png",".jpg",".jpeg"]:
            ip = f"{DATA}/{split}/images/{stem}{ext}"
            if os.path.exists(ip): break
        else: continue
        img = Image.open(ip).convert("RGB"); W,H = img.size
        for line in open(f"{DATA}/{split}/labels/{lf}"):
            if not line.strip(): continue
            c,xc,yc,bw,bh = map(float, line.split()); c=int(c)
            if per_class[c] >= MAX_CROPS_PER_CLASS: continue
            w,h = bw*W, bh*H
            if w<16 or h<16: continue
            pad=0.15
            box = (max(0,(xc-bw/2)*W-pad*w), max(0,(yc-bh/2)*H-pad*h),
                   min(W,(xc+bw/2)*W+pad*w), min(H,(yc+bh/2)*H+pad*h))
            X.append(np.asarray(img.crop(box).resize((size,size)), dtype=np.uint8)); y.append(c)
            per_class[c]+=1
    return np.array(X), np.array(y)

crops = {sp: build_crops(sp, CROP_SIZE) for sp in ["train","val","test"]}
for sp in crops:
    print(sp, crops[sp][0].shape, {CLASSES[i]: int(v) for i,v in enumerate(np.bincount(crops[sp][1], minlength=NUM_CLS))})
(Xtr,ytr),(Xva,yva),(Xte,yte) = crops["train"], crops["val"], crops["test"]

from sklearn.utils.class_weight import compute_class_weight
present_cls = np.unique(ytr)
w = compute_class_weight("balanced", classes=present_cls, y=ytr)
CLASS_WEIGHT = {int(c): float(v) for c, v in zip(present_cls, w)}
CW_TENSOR = np.ones(NUM_CLS, dtype="float32")
for c, v in CLASS_WEIGHT.items(): CW_TENSOR[c] = v
print("Class weights:", {CLASSES[c]: round(v,2) for c,v in CLASS_WEIGHT.items()})

from sklearn.metrics import accuracy_score, f1_score, classification_report
def multiclass_metrics(y_true, pred):
    acc = accuracy_score(y_true, pred)
    macro = f1_score(y_true, pred, average="macro")
    per_cls = f1_score(y_true, pred, average=None, labels=list(range(NUM_CLS)), zero_division=0)
    support = np.bincount(y_true, minlength=NUM_CLS)
    valid = [c for c in range(NUM_CLS) if support[c] >= 10]   # lớp <10 mẫu (vd no_boots) loại khỏi worst - báo cáo riêng
    worst = float(np.min([per_cls[c] for c in valid])) if valid else 0.0
    return acc, macro, worst

## 1. HPO vòng 1: cell × hidden (8 cấu hình × 6 epochs, chọn theo macro-F1 VAL)

In [ ]:
import tensorflow as tf
for g in tf.config.list_physical_devices('GPU'): tf.config.experimental.set_memory_growth(g, True)
from tensorflow import keras
from tensorflow.keras import layers

EP_SCREEN = 2 if QUICK_TEST else 6
EP_FINAL  = 3 if QUICK_TEST else 15

Xtr_s = (Xtr.astype("float32")/255.).reshape(len(Xtr), CROP_SIZE, CROP_SIZE*3)
Xva_s = (Xva.astype("float32")/255.).reshape(len(Xva), CROP_SIZE, CROP_SIZE*3)
Xte_s = (Xte.astype("float32")/255.).reshape(len(Xte), CROP_SIZE, CROP_SIZE*3)

def rnn_layer(cell, hidden, dropout, return_sequences=False):
    if cell=="GRU":    return layers.GRU(hidden, dropout=dropout, return_sequences=return_sequences)
    if cell=="LSTM":   return layers.LSTM(hidden, dropout=dropout, return_sequences=return_sequences)
    if cell=="BiGRU":  return layers.Bidirectional(layers.GRU(hidden, dropout=dropout, return_sequences=return_sequences))
    return layers.Bidirectional(layers.LSTM(hidden, dropout=dropout, return_sequences=return_sequences))

def make_rnn(cell, hidden, dropout=0.2, n_layers=1, opt="adam"):
    inp = layers.Input((CROP_SIZE, CROP_SIZE*3)); x = inp
    for li in range(n_layers):
        x = rnn_layer(cell, hidden, dropout, return_sequences=(li < n_layers-1))(x)
    x = layers.Dense(64, activation="relu")(x)
    m = keras.Model(inp, layers.Dense(NUM_CLS, activation="softmax")(x))
    optimizer = keras.optimizers.Adam(1e-3) if opt=="adam" else keras.optimizers.RMSprop(1e-3)
    m.compile(optimizer=optimizer, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m
def cbs(): return [keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]

def screen(cfg, key):
    if done(key): print("skip:", STATE[key]); return STATE[key]
    m = make_rnn(**cfg); t0=time.time()
    m.fit(Xtr_s, ytr, validation_data=(Xva_s,yva), epochs=EP_SCREEN, batch_size=64,
          class_weight=CLASS_WEIGHT, verbose=0)
    pred = m.predict(Xva_s, verbose=0).argmax(1)
    acc, macro, worst = multiclass_metrics(yva, pred)
    row = {**cfg, "macro_f1_val": round(macro,4), "min": round((time.time()-t0)/60,1)}
    mark(key, row); print(row); keras.backend.clear_session(); return row

GRID1 = [{"cell":c,"hidden":h} for c in ["GRU","LSTM","BiGRU","BiLSTM"] for h in [128,256]]
rows1 = [screen(cfg, f"rnn_g1_{i}") for i, cfg in enumerate(GRID1)] if RUN_HPO else []
if RUN_HPO:
    hpo1 = pd.DataFrame(rows1).sort_values("macro_f1_val", ascending=False)
    HPO_TABLES["RNN_round1_cell_hidden"] = hpo1; display(hpo1)
    best1 = {"cell": hpo1.iloc[0]["cell"], "hidden": int(hpo1.iloc[0]["hidden"])}
else:
    best1 = {"cell":"BiLSTM","hidden":128}
print("Vòng 1 chọn:", best1)

## 2. HPO vòng 2 & 3: dropout × optimizer, rồi stacked 2 tầng (trên cấu hình thắng)

In [ ]:
if RUN_HPO:
    GRID2 = [{**best1, "dropout":d, "opt":o} for d in [0.2,0.4] for o in ["adam","rmsprop"]]
    rows2 = [screen(cfg, f"rnn_g2_{i}") for i, cfg in enumerate(GRID2)]
    hpo2 = pd.DataFrame(rows2).sort_values("macro_f1_val", ascending=False)
    HPO_TABLES["RNN_round2_dropout_opt"] = hpo2; display(hpo2)
    best2 = {**best1, "dropout": float(hpo2.iloc[0]["dropout"]), "opt": hpo2.iloc[0]["opt"]}
    row3 = screen({**best2, "n_layers":2}, "rnn_g3_stacked")
    best_cfg = {**best2, "n_layers": 2} if row3["macro_f1_val"] > hpo2.iloc[0]["macro_f1_val"] else {**best2, "n_layers": 1}
else:
    best_cfg = {"cell":"BiLSTM","hidden":128,"dropout":0.2,"opt":"adam","n_layers":1}
print("Cấu hình RNN cuối:", best_cfg)

## 3. Train cuối + đánh giá TEST (1 lần duy nhất)

In [ ]:
res_cls = {}
rnn_name = f"{best_cfg['cell']}-{best_cfg['hidden']}x{best_cfg['n_layers']} (RNN)"
if done("rnn_final"):
    res_cls[STATE["rnn_final"]["_name"]] = STATE["rnn_final"]; print("skip:", STATE["rnn_final"])
else:
    m = make_rnn(**best_cfg); t0=time.time()
    m.fit(Xtr_s, ytr, validation_data=(Xva_s,yva), epochs=EP_FINAL, batch_size=64,
          class_weight=CLASS_WEIGHT, callbacks=cbs(), verbose=2)
    pred = m.predict(Xte_s, verbose=0).argmax(1)
    acc, macro, worst = multiclass_metrics(yte, pred)
    res_cls[rnn_name] = {"_name": rnn_name, "cfg": str(best_cfg),
        "accuracy":round(acc,4), "macro_f1":round(macro,4), "worst_class_f1":round(worst,4),
        "train_min":round((time.time()-t0)/60,1)}
    print(res_cls[rnn_name])
    print(classification_report(yte, pred, labels=list(range(NUM_CLS)), target_names=CLASSES, zero_division=0))
    m.save(f"{WORK}/results/rnn_best.keras")
    mark("rnn_final", res_cls[rnn_name]); keras.backend.clear_session()

# Tham chiếu thêm: 3 cell chuẩn (GRU/LSTM/BiLSTM) cùng hidden thắng - để có bảng so sánh cell đầy đủ trên TEST
for cell_name in ["GRU","LSTM","BiLSTM"]:
    key = f"rnn_ref_{cell_name}"
    ref_name = f"{cell_name}-ref (RNN)"
    if done(key): res_cls[STATE[key]["_name"]] = STATE[key]; continue
    m = make_rnn(cell=cell_name, hidden=best_cfg["hidden"], dropout=best_cfg["dropout"], opt=best_cfg["opt"]); t0=time.time()
    m.fit(Xtr_s, ytr, validation_data=(Xva_s,yva), epochs=EP_FINAL, batch_size=64,
          class_weight=CLASS_WEIGHT, callbacks=cbs(), verbose=0)
    pred = m.predict(Xte_s, verbose=0).argmax(1)
    acc, macro, worst = multiclass_metrics(yte, pred)
    res_cls[ref_name] = {"_name": ref_name, "cfg": f"{cell_name}, hidden {best_cfg['hidden']}",
        "accuracy":round(acc,4), "macro_f1":round(macro,4), "worst_class_f1":round(worst,4),
        "train_min":round((time.time()-t0)/60,1)}
    mark(key, res_cls[ref_name]); print(res_cls[ref_name]); keras.backend.clear_session()

## 4. Xuất kết quả chuẩn hóa

In [ ]:
# ====== XUẤT summary.csv chuẩn hóa để notebook 04 tổng hợp ======
rows = []
for name, d in res_cls.items():
    rows.append({"branch": BRANCH, "task": "classification", "model": name,
                 **{k: v for k, v in d.items() if k != "_name"}})
df_sum = pd.DataFrame(rows)
df_sum.to_csv(f"{WORK}/results/summary.csv", index=False)
for name, tbl in HPO_TABLES.items():
    tbl.to_csv(f"{WORK}/results/hpo_{name.replace('/','').replace(' ','_')}.csv", index=False)
print("Đã lưu results/summary.csv + các bảng HPO. Sau khi COMMIT xong, dùng output này làm Input cho notebook 04.")
display(df_sum)

## Checkpoint & Resume (nếu notebook này bị cắt trước khi xong)
Notebook lưu tiến độ vào `state.json` + `runs/` sau mỗi cấu hình/model. Nếu phiên bị cắt: (1) chạy bằng **Save & Run All (Commit)** nên output vẫn được giữ trong version đó; (2) mở notebook → **+ Add Input → Your Work** → chọn output version vừa dừng; (3) đặt `RESUME_FROM = "/kaggle/input/<slug>"` ở cell Config; (4) Run All lại — stage đã xong tự bỏ qua, detection train tiếp từ `last.pt`.